# Notebook 07 - Preparação dos Dados ABIDE 
**Objetivo:** Preparar as características de conectividade (`X`), os rótulos (`y`) e os grupos (`SITE_ID`) para a etapa de classificação por Machine Learning.

**Entradas:** `Data/conectivity/connectivity_matrices_abide_validado.pkl` (Caderno 05) e `Data/raw/phenotypic_abide.csv` (Caderno 02).

**Processamento:** Extração das conexões únicas das matrizes de conectividade 200 × 200, resultando em 19.900 características por participante. Os rótulos são recodificados para `0 = Controle` e `1 = TEA`. A validação cruzada é definida utilizando `StratifiedGroupKFold`, com 5 folds e `SITE_ID` como grupo.

O `StandardScaler` não é ajustado neste notebook. Qualquer transformação que aprenda parâmetros a partir dos dados deve ser ajustada exclusivamente no conjunto de treinamento de cada fold, no notebook de classificação, para evitar data leakage.

**Saída:** `Data/processed/dataset_abide_processado.pkl`.

In [1]:
import sys
from pathlib import Path

_raiz = Path.cwd().resolve()
for _cand in (_raiz, *_raiz.parents):
    if (_cand / 'src' / 'config.py').exists():
        _raiz = _cand
        break
else:
    raise RuntimeError(
        'Não foi possível localizar a pasta src/config.py a partir de '
        f'{Path.cwd()}. Abra a pasta TCC_TEA_TDAH/ no VS Code.'
    )
if str(_raiz) not in sys.path:
    sys.path.insert(0, str(_raiz))

import os
import random
import numpy as np

from src.config import BASE_DIR, PATHS, SEED, criar_pastas
from src.utils import exigir_arquivo

criar_pastas()

random.seed(SEED)
np.random.seed(SEED)

print('Projeto (BASE_DIR) =', BASE_DIR)

Projeto (BASE_DIR) = /home/maria/TCC_TEA


In [2]:
# ============================================================
# CARREGAMENTO E PREPARAÇÃO DAS CARACTERÍSTICAS
# ============================================================

exigir_arquivo(
    PATHS['conectivity']
    / 'connectivity_matrices_abide_validado.pkl',
    '05_Validacao_ABIDE.ipynb'
)

exigir_arquivo(
    PATHS['raw']
    / 'phenotypic_abide.csv',
    '02_Download_ABIDE.ipynb'
)

import pickle
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Carregar matrizes de conectividade validadas
# ------------------------------------------------------------
conn_path = (
    PATHS['conectivity']
    / 'connectivity_matrices_abide_validado.pkl'
)

with open(conn_path, 'rb') as f:
    conn_dict = pickle.load(f)

conn_dict = {
    str(sub_id): np.asarray(mat)
    for sub_id, mat in conn_dict.items()
}

# ------------------------------------------------------------
# Carregar dados fenotípicos
# ------------------------------------------------------------
phenotypic = pd.read_csv(
    PATHS['raw'] / 'phenotypic_abide.csv'
)

phenotypic['SUB_ID'] = (
    phenotypic['SUB_ID']
    .astype(str)
)

phenotypic = phenotypic.set_index(
    'SUB_ID'
)

# ------------------------------------------------------------
# IDs dos participantes
# ------------------------------------------------------------
sub_ids = np.array(
    list(conn_dict.keys()),
    dtype=str
)

# ------------------------------------------------------------
# Conferir correspondência dos participantes
# ------------------------------------------------------------
ids_sem_fenotipo = [
    sub_id
    for sub_id in sub_ids
    if sub_id not in phenotypic.index
]

if ids_sem_fenotipo:
    raise ValueError(
        'Existem participantes com matriz de conectividade '
        'sem correspondente nos dados fenotípicos. '
        f'Quantidade: {len(ids_sem_fenotipo)}'
    )

# ------------------------------------------------------------
# Matrizes de conectividade
# ------------------------------------------------------------
matrices = np.stack(
    [conn_dict[sub_id] for sub_id in sub_ids]
)

# ------------------------------------------------------------
# Conferência da dimensão das matrizes
# ------------------------------------------------------------
if matrices.ndim != 3:
    raise ValueError(
        f'Formato inesperado das matrizes: {matrices.shape}'
    )

if matrices.shape[1:] != (200, 200):
    raise ValueError(
        'As matrizes de conectividade devem possuir '
        f'dimensão 200 × 200. Encontrado: {matrices.shape[1:]}'
    )

# ------------------------------------------------------------
# Extração das conexões únicas
#
# Matriz simétrica:
# 200 × 200
#
# Número de conexões sem diagonal:
# 200 × 199 / 2 = 19.900
# ------------------------------------------------------------
# Triângulo SUPERIOR sem a diagonal, em ordem de linha (mesma convenção do TDAH).
# Atenção: nilearn.sym_matrix_to_vec devolve o triângulo INFERIOR; a ordem dos 19.900 atributos
# é diferente. Como os notebooks 09/10 reconstroem a matriz a partir de X, a ordem precisa ser conhecida.
roi_i, roi_j = np.triu_indices(matrices.shape[1], k=1)
X = matrices[:, roi_i, roi_j].astype(np.float64)

# ------------------------------------------------------------
# Rótulos
#
# DX_GROUP:
# 1 = TEA
# 2 = Controle
# ------------------------------------------------------------
y_original = phenotypic.loc[
    sub_ids,
    'DX_GROUP'
].to_numpy()

# ------------------------------------------------------------
# Grupos para validação
#
# SITE_ID representa o site de aquisição.
# ------------------------------------------------------------
groups = phenotypic.loc[
    sub_ids,
    'SITE_ID'
].astype(str)
groups = phenotypic.loc[
    sub_ids,
    'SITE_ID'
].astype(str).to_numpy()

# ------------------------------------------------------------
# Conferências finais
# ------------------------------------------------------------
if X.shape[1] != 19900:
    raise ValueError(
        'A quantidade de características não é 19.900. '
        f'Encontrado: {X.shape[1]}'
    )

if len(X) != len(y_original) or len(X) != len(groups):
    raise ValueError(
        'X, y e groups possuem quantidades diferentes '
        'de participantes.'
    )

print('==========================================')
print('DADOS PREPARADOS')
print('==========================================')
print('Participantes:', len(sub_ids))
print('X:', X.shape)
print('y original:', y_original.shape)
print('Sites únicos:', len(np.unique(groups)))
print('Características por participante:', X.shape[1])
print('Esperado:', 19900)

DADOS PREPARADOS
Participantes: 871
X: (871, 19900)
y original: (871,)
Sites únicos: 20
Características por participante: 19900
Esperado: 19900


In [3]:

# ============================================================
# AUDITORIA DAS MATRIZES + FISHER Z (aplicado uma única vez)
# ============================================================
n_part = matrices.shape[0]
diag = np.diagonal(matrices, axis1=1, axis2=2)
print('AUDITORIA DAS MATRIZES (Pearson, antes do Fisher z)')
print('  Formato:', matrices.shape)
print('  Finitas:', bool(np.isfinite(matrices).all()))
print('  Simétricas (atol=1e-6):', bool(np.allclose(matrices, np.swapaxes(matrices, 1, 2), atol=1e-6)))
print('  Intervalo:', float(matrices.min()), 'a', float(matrices.max()))
print('  Diagonal: mín=%.4f máx=%.4f (a diagonal é DESCARTADA na vetorização)' % (diag.min(), diag.max()))
if not np.isfinite(matrices).all():
    raise ValueError('Matrizes com NaN/infinito.')
if np.any(matrices < -1.000001) or np.any(matrices > 1.000001):
    raise ValueError('Há valores fora de [-1, 1].')

# Trava contra aplicação dupla: Pearson está em [-1, 1]; Fisher z já aplicado passaria de 1.
if np.abs(X).max() > 1.000001:
    raise ValueError('X já parece estar em Fisher z (|X| > 1). Não aplique de novo.')

EPS = 1e-7
X_pearson = X.copy()
X = np.arctanh(np.clip(X_pearson, -1.0 + EPS, 1.0 - EPS)).astype(np.float32)
if not np.isfinite(X).all():
    raise ValueError('X contém NaN/infinito após Fisher z.')
print('\nFisher z aplicado: sim (uma vez).')
print('  X:', X.shape, '| intervalo após Fisher z: %.3f a %.3f' % (X.min(), X.max()))
print('  Correlação entre r e z por participante (deve ser ~1, transformação monótona): %.4f' %
      np.corrcoef(X_pearson[0], X[0])[0, 1])


AUDITORIA DAS MATRIZES (Pearson, antes do Fisher z)
  Formato: (871, 200, 200)
  Finitas: True
  Simétricas (atol=1e-6): True
  Intervalo: -0.7995274548218834 a 1.0
  Diagonal: mín=1.0000 máx=1.0000 (a diagonal é DESCARTADA na vetorização)

Fisher z aplicado: sim (uma vez).
  X: (871, 19900) | intervalo após Fisher z: -1.097 a 1.922
  Correlação entre r e z por participante (deve ser ~1, transformação monótona): 0.9900


In [4]:
# ============================================================
# RECODIFICAÇÃO DOS RÓTULOS
# ============================================================
#
# DX_GROUP:
# 1 = TEA
# 2 = Controle
#
# Para os modelos:
# 0 = Controle
# 1 = TEA
# ============================================================

y_bin = (
    y_original == 1
).astype(int)

valores_validos = set(
    np.unique(y_original)
)

if not valores_validos.issubset({1, 2}):
    raise ValueError(
        'Foram encontrados valores inesperados em DX_GROUP: '
        f'{sorted(valores_validos)}'
    )

print('==========================================')
print('DISTRIBUIÇÃO DAS CLASSES')
print('==========================================')
print(
    'Controle (0):',
    int(np.sum(y_bin == 0))
)
print(
    'TEA (1):',
    int(np.sum(y_bin == 1))
)
print(
    'Total:',
    len(y_bin)
)

DISTRIBUIÇÃO DAS CLASSES
Controle (0): 468
TEA (1): 403
Total: 871


In [5]:

# ============================================================
# FOLDS EXTERNOS (preservar os já auditados; não regerar de forma aleatória)
# ============================================================
from sklearn.model_selection import StratifiedGroupKFold
import shutil

N_SPLITS = 5
TAMANHOS_AUDITADOS = [41, 173, 165, 324, 168]     # tamanhos dos folds de teste vistos na saída anterior
out_path = PATHS['processed'] / 'dataset_abide_processado.pkl'
backup_path = PATHS['processed'] / 'dataset_abide_processado_pearson_antigo.pkl'

folds = None
reutilizados = False
if out_path.exists():
    with open(out_path, 'rb') as f:
        antigo = pickle.load(f)
    mesmos_ids = np.array_equal(np.asarray(antigo['sub_ids']).astype(str), sub_ids.astype(str))
    mesmos_sites = np.array_equal(np.asarray(antigo['groups']).astype(str), groups.astype(str))
    if mesmos_ids and mesmos_sites:
        folds = [(np.asarray(a), np.asarray(b)) for a, b in antigo['folds']]
        if not backup_path.exists():
            shutil.copy(out_path, backup_path)
        reutilizados = True
        print('Folds REUTILIZADOS do arquivo anterior (mesmos participantes e sites).')
        print('Cópia do arquivo antigo (Pearson) em:', backup_path)
    else:
        print('ATENÇÃO: o arquivo anterior tem outra lista de participantes/sites; os folds serão recriados.')

if folds is None:
    sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    folds = list(sgkf.split(X, y_bin, groups))
    print('Folds RECRIADOS com StratifiedGroupKFold (semente %d).' % SEED)

fold_id = np.full(len(y_bin), -1, dtype=np.int64)
for k, (tr_idx, te_idx) in enumerate(folds):
    fold_id[te_idx] = k
if np.any(fold_id < 0):
    raise RuntimeError('Há participantes sem fold de teste.')

print('==========================================')
print('VALIDAÇÃO DOS FOLDS')
print('==========================================')
tamanhos = []
for k, (tr_idx, te_idx) in enumerate(folds):
    inter = set(groups[tr_idx]) & set(groups[te_idx])
    cls_te = np.unique(y_bin[te_idx]); cls_tr = np.unique(y_bin[tr_idx])
    tamanhos.append(len(te_idx))
    print(f'Fold {k}: treino={len(tr_idx)} teste={len(te_idx)} | controles/TEA no teste = '
          f'{int((y_bin[te_idx]==0).sum())}/{int((y_bin[te_idx]==1).sum())} | sites em comum = {len(inter)} | '
          f'sites no teste: {sorted(set(groups[te_idx]))}')
    if inter:
        raise RuntimeError(f'Fold {k}: site em treino e teste.')
    if len(cls_te) < 2 or len(cls_tr) < 2:
        raise RuntimeError(f'Fold {k}: falta uma classe.')
print('\nTamanhos de teste encontrados:', tamanhos, '| auditados:', TAMANHOS_AUDITADOS)
if sorted(tamanhos) != sorted(TAMANHOS_AUDITADOS):
    if reutilizados:
        print('ATENÇÃO: os folds reutilizados NÃO têm os tamanhos auditados (41/173/165/324/168). '
              'Investigue: o arquivo anterior pode ter sido gerado com outros dados.')
    else:
        print('ATENÇÃO: os folds recriados diferem dos auditados (41/173/165/324/168), o que pode acontecer se o '
              'StratifiedGroupKFold mudou entre versões do scikit-learn. Se aceitar estes novos folds, refaça TODOS os modelos com eles.')
print('Sites distintos:', len(np.unique(groups)))
print('Sites com uma só classe:', [s for s in np.unique(groups) if len(np.unique(y_bin[groups == s])) < 2])


Folds RECRIADOS com StratifiedGroupKFold (semente 42).
VALIDAÇÃO DOS FOLDS
Fold 0: treino=830 teste=41 | controles/TEA no teste = 19/22 | sites em comum = 0 | sites no teste: ['YALE']
Fold 1: treino=698 teste=173 | controles/TEA no teste = 101/72 | sites em comum = 0 | sites no teste: ['KKI', 'OLIN', 'SBL', 'UM_1']
Fold 2: treino=706 teste=165 | controles/TEA no teste = 91/74 | sites em comum = 0 | sites no teste: ['CALTECH', 'LEUVEN_1', 'LEUVEN_2', 'PITT', 'TRINITY']
Fold 3: treino=547 teste=324 | controles/TEA no teste = 175/149 | sites em comum = 0 | sites no teste: ['CMU', 'NYU', 'OHSU', 'SDSU', 'STANFORD', 'UCLA_1']
Fold 4: treino=703 teste=168 | controles/TEA no teste = 82/86 | sites em comum = 0 | sites no teste: ['MAX_MUN', 'UCLA_2', 'UM_2', 'USM']

Tamanhos de teste encontrados: [41, 173, 165, 324, 168] | auditados: [41, 173, 165, 324, 168]
Sites distintos: 20
Sites com uma só classe: []


In [6]:

# ============================================================
# SALVAR DATASET PROCESSADO (Fisher z)
# ============================================================
dataset_processado = {
    'X': X,                      # Fisher z, 19.900 conexões (triângulo superior sem diagonal)
    'y': y_bin,                  # 0 = Controle, 1 = TEA
    'groups': groups,            # SITE_ID
    'sub_ids': sub_ids,
    'folds': folds,              # lista de (treino, teste), como antes
    'fold_id': fold_id,          # fold de teste de cada participante
    'n_splits': N_SPLITS,
    'seed': SEED,
    'roi_i': roi_i.astype(np.int16),
    'roi_j': roi_j.astype(np.int16),
    'ordem_atributos': 'triu_linha',
    'fisher_z': True,
    'eps_fisher': EPS,
}
with open(out_path, 'wb') as f:
    pickle.dump(dataset_processado, f, protocol=pickle.HIGHEST_PROTOCOL)

print('==========================================')
print('DATASET PROCESSADO SALVO')
print('==========================================')
print('Arquivo:', out_path)
print('X:', X.shape, '| Fisher z: sim | folds:', len(folds), '| seed:', SEED)
print('O StandardScaler NÃO é ajustado aqui: fica dentro dos pipelines de cada fold (notebooks 08/09/10).')


DATASET PROCESSADO SALVO
Arquivo: /home/maria/TCC_TEA/Data/processed/dataset_abide_processado.pkl
X: (871, 19900) | Fisher z: sim | folds: 5 | seed: 42
O StandardScaler NÃO é ajustado aqui: fica dentro dos pipelines de cada fold (notebooks 08/09/10).
